# 4.1 训练目标 (Training Objectives)

训练目标定义了模型从数据中学习什么，是预训练阶段最核心的设计选择。

本节涵盖：
- 因果语言建模（CLM）
- 掩码语言建模（MLM）
- 前缀语言建模
- 去噪自编码

## 1. 因果语言建模（CLM）

**目的**：自回归地预测下一个token

**基本原理**：给定前文token序列，模型预测下一个token的概率分布，使用交叉熵损失优化。自然语言本身提供了"正确答案"——下一个词就是标签。

**核心公式**：
- L_CLM = -Σ log P(x_t | x_{<t})
- 每个位置t的标签就是x_t本身
- 因果掩码确保模型只能看到t之前的token

**代表模型**：GPT系列、LLaMA、Mistral、Qwen

**特点**：天然适合生成任务，是当前大语言模型的主流预训练目标

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class MiniCausalModel(nn.Module):
    def __init__(self, vocab_size=500, d_model=64, n_heads=2, n_layers=1, max_seq_len=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        T = x.size(1)
        h = self.embed(x) + self.pos_embed(torch.arange(T, device=x.device))
        h = self.encoder(h, mask=mask)
        return self.head(h)

model = MiniCausalModel()
X = torch.randint(0, 500, (4, 32))

causal_mask = nn.Transformer.generate_square_subsequent_mask(32)
logits = model(X, mask=causal_mask)

shift_logits = logits[:, :-1, :].reshape(-1, 500)
shift_labels = X[:, 1:].reshape(-1)
clm_loss = F.cross_entropy(shift_logits, shift_labels)

print('=== Causal Language Modeling (CLM) ===')
print(f'Input tokens: {X.shape}')
print(f'Output logits: {logits.shape}')
print(f'CLM Loss: {clm_loss.item():.4f}')

with torch.no_grad():
    probs = F.softmax(logits, dim=-1)
    top1_preds = probs.argmax(dim=-1)
    accuracy = (top1_preds[:, :-1] == X[:, 1:]).float().mean()
print(f'Random baseline accuracy: {1/500:.4f}')
print(f'Model top-1 accuracy: {accuracy:.4f} (untrained, should be near random)')

print(f'\nKey: CLM shifts labels by 1 position.')
print(f'Input: [x0, x1, x2, ..., xT-1]')
print(f'Target: [x1, x2, x3, ..., xT]')
print(f'Each position predicts the NEXT token using only past context.')

## 2. 掩码语言建模（MLM）

**目的**：通过预测被遮蔽的token学习双向表征

**基本原理**：随机遮蔽输入中15%的token，让模型根据双向上下文预测被遮蔽的内容。被遮蔽的token就是天然的标签。

**MLM策略（BERT风格）**：
- 80%的遮蔽位置替换为[MASK]token
- 10%替换为随机token
- 10%保持不变
- 仅在遮蔽位置计算损失

**代表模型**：BERT、RoBERTa、DeBERTa

**与CLM的区别**：MLM利用双向上下文，理解能力更强；CLM是自回归的，生成能力更强

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

model = MiniCausalModel()
X = torch.randint(0, 500, (4, 32))

mask_prob = 0.15
mask_positions = torch.bernoulli(torch.full(X.shape, mask_prob)).bool()

X_masked = X.clone()
X_masked[mask_positions] = 499

logits_bi = model(X_masked)

labels = X.clone()
labels[~mask_positions] = -100

mlm_loss = F.cross_entropy(logits_bi.reshape(-1, 500), labels.reshape(-1), ignore_index=-100)

n_masked = mask_positions.sum().item()
n_total = X.numel()

print('=== Masked Language Modeling (MLM) ===')
print(f'Total tokens: {n_total}')
print(f'Masked tokens: {n_masked} ({n_masked/n_total:.1%})')
print(f'MLM Loss (on masked positions only): {mlm_loss.item():.4f}')

with torch.no_grad():
    masked_logits = logits_bi[mask_positions]
    masked_labels = X[mask_positions]
    top1 = masked_logits.argmax(dim=-1)
    acc = (top1 == masked_labels).float().mean()
print(f'Masked token accuracy: {acc:.4f} (untrained, ~0.2% random baseline)')

print(f'\nKey: MLM uses BIDIRECTIONAL context to predict masked tokens.')
print(f'This gives stronger understanding but cannot be used for autoregressive generation.')

## 3. 前缀语言建模

**目的**：兼顾理解和生成

**基本原理**：对输入前缀部分使用双向注意力，对生成部分使用因果注意力，适用于编码器-解码器架构。

**核心设计**：
- 前缀部分：双向注意力，完整理解输入
- 生成部分：因果掩码，自回归生成
- 同一模型同时支持理解和生成任务

**代表模型**：UL2、PaLM（prefix-LM模式）

**优势**：统一了CLM和MLM的优点，一个模型同时擅长理解和生成

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

seq_len = 16
prefix_len = 8

prefix_mask = torch.ones(seq_len, seq_len)
prefix_mask[:prefix_len, :prefix_len] = 1
prefix_mask[prefix_len:, prefix_len:] = torch.tril(torch.ones(seq_len - prefix_len, seq_len - prefix_len))
prefix_mask[prefix_len:, :prefix_len] = 1

causal_mask = torch.tril(torch.ones(seq_len, seq_len))

print('=== Prefix Language Modeling ===')
print(f'Sequence length: {seq_len}, Prefix length: {prefix_len}')
print(f'\nPrefix attention mask (1=attend, 0=mask):')
print(prefix_mask.int().tolist())
print(f'\nCausal attention mask (for comparison):')
print(causal_mask.int().tolist())

print(f'\nKey differences:')
print(f'  Prefix positions 0-{prefix_len-1}: BIDIRECTIONAL attention (see all prefix tokens)')
print(f'  Generation positions {prefix_len}-{seq_len-1}: CAUSAL attention (see prefix + past generation)')
print(f'\nThis combines understanding (bidirectional) with generation (autoregressive).')

## 4. 去噪自编码

**目的**：学习从损坏输入恢复原始文本

**基本原理**：对输入文本施加噪声（如随机删除、插入、替换token span），训练模型恢复原始文本。原始文本本身就是监督信号。

**典型噪声策略**：
- **Span corruption**（T5）：随机遮蔽不同长度的token span，每个span替换为一个哨兵token
- **Token infilling**（BART）：随机删除token或打乱句子顺序

**代表模型**：T5、BART

**与MLM的区别**：MLM遮蔽单个token，去噪自编码遮蔽连续的span，需要模型理解更长的上下文来恢复

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(42)

def span_corrupt(tokens, vocab_size, mask_token_id=498, sentinel_start=490, corrupt_ratio=0.15, mean_span_len=3):
    n_tokens = len(tokens)
    n_corrupt = max(1, int(n_tokens * corrupt_ratio))
    n_spans = max(1, n_corrupt // mean_span_len)

    span_lengths = []
    remaining = n_corrupt
    for i in range(n_spans):
        if i == n_spans - 1:
            span_lengths.append(remaining)
        else:
            l = random.randint(1, min(mean_span_len * 2, remaining - (n_spans - i - 1)))
            span_lengths.append(l)
            remaining -= l

    positions = sorted(random.sample(range(n_tokens - max(span_lengths)), n_spans))

    corrupted = tokens.clone()
    target_parts = []
    sentinel_id = sentinel_start

    for i, (pos, length) in enumerate(zip(positions, span_lengths)):
        target_parts.append(torch.tensor([sentinel_id]))
        target_parts.append(tokens[pos:pos+length])
        corrupted = torch.cat([corrupted[:pos], torch.tensor([sentinel_id]), corrupted[pos+length:]])
        sentinel_id += 1

    target_parts.append(torch.tensor([sentinel_id]))
    target = torch.cat(target_parts)

    return corrupted, target

tokens = torch.randint(0, 490, (32,))
corrupted, target = span_corrupt(tokens, 500)

print('=== Denoising Autoencoding (Span Corruption) ===')
print(f'Original tokens ({len(tokens)}): {tokens[:10].tolist()}...')
print(f'Corrupted tokens ({len(corrupted)}): {corrupted[:12].tolist()}...')
print(f'Target output ({len(target)}): {target.tolist()}')

sentinel_positions = (corrupted >= 490).nonzero().squeeze(-1).tolist()
print(f'\nSentinel tokens in input at positions: {sentinel_positions}')
print(f'\nKey: Model learns to reconstruct original spans from sentinel markers.')
print(f'Sentinel <s_i> in input -> <s_i> + original_span in output')
print(f'This is the T5 pretraining objective.')